In [32]:
!pip install google-play-scraper pandas

In [33]:
!pip install Sastrawi tqdm
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [34]:
import pandas as pd
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords
from tqdm import tqdm

# Mengaktifkan progress bar untuk pandas
tqdm.pandas()

# Memuat dataset yang baru saja kita scrape
df = pd.read_csv('dataset_ulasan_hotel_traveloka.csv')

# Menyiapkan alat pembersih teks
factory = StemmerFactory()
stemmer = factory.create_stemmer()
stop_words = set(stopwords.words('indonesian'))

# Menambahkan beberapa kata tambahan yang sering muncul tapi tidak bermakna sentimen
custom_stopwords = {'yg', 'nya', 'di', 'ke', 'dari', 'buat', 'untuk', 'aja', 'dan', 'ini', 'itu', 'sih', 'dong'}
stop_words.update(custom_stopwords)

def bersihkan_teks(text):
    if type(text) != str:
        return ""

    # 1. Case Folding: Ubah semua huruf menjadi kecil
    text = text.lower()

    # 2. Cleaning: Hapus URL, username, angka, dan tanda baca (sisakan huruf a-z saja)
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 3. Tokenisasi, Stopword Removal, dan Stemming
    words = text.split()
    cleaned_words = [stemmer.stem(word) for word in words if word not in stop_words]

    # Gabungkan kembali menjadi satu kalimat utuh
    return ' '.join(cleaned_words)

print("Memulai proses pembersihan teks besar-besaran...")
# Terapkan fungsi pembersih ke kolom 'content' dan simpan hasilnya di kolom baru
df['clean_content'] = df['content'].progress_apply(bersihkan_teks)

# Menghapus baris yang mungkin menjadi kosong setelah dibersihkan
df.replace('', pd.NA, inplace=True)
df.dropna(subset=['clean_content'], inplace=True)

# Simpan dataset yang sudah bersih
file_bersih = 'dataset_traveloka_bersih.csv'
df.to_csv(file_bersih, index=False, encoding='utf-8')

print(f"\nSelesai! Data berhasil dibersihkan dan disimpan ke '{file_bersih}'")
# Tampilkan perbandingan 5 data pertama
print(df[['content', 'clean_content']].head())

Memulai proses pembersihan teks besar-besaran...


100%|██████████| 10000/10000 [04:55<00:00, 33.79it/s]


Selesai! Data berhasil dibersihkan dan disimpan ke 'dataset_traveloka_bersih.csv'
                                             content  \
0                                          apk tolol   
1                                  sangat baik skali   
2                                              bagus   
3  aplikasi ngaco, ngajukan refund tiket awalnya ...   
4  saya kasih bintang 1 karena sangat mengecewaka...   

                                       clean_content  
0                                          apk tolol  
1                                              skali  
2                                              bagus  
3  aplikasi ngaco ngajukan refund tiket tuju tola...  
4  kasih bintang kecewa tpaylater g tdk bayar tel...  


In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Persiapan Data
df = pd.read_csv('dataset_traveloka_bersih.csv')
df.dropna(subset=['clean_content', 'rating'], inplace=True)

def pelabelan(rating):
    if rating <= 2: return 'Negatif'
    elif rating == 3: return 'Netral'
    else: return 'Positif'

df['label_sentimen'] = df['rating'].apply(pelabelan)

# 2. Skema Pembagian Data 80/20 & Ekstraksi Fitur TF-IDF
X = df['clean_content']
y = df['label_sentimen']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Dictionary untuk menyimpan 3 skema model
models = {
    "Skema 1 (Naive Bayes)": MultinomialNB(),
    "Skema 2 (Random Forest)": RandomForestClassifier(n_estimators=100, random_state=42),
    "Skema 3 (Deep Learning - MLP)": MLPClassifier(hidden_layer_sizes=(16,), max_iter=100, random_state=42)
}

# 3. Eksekusi dan Evaluasi 3 Skema
print("="*60)
for name, model in models.items():
    print(f"Melatih {name}...")
    model.fit(X_train_vec, y_train)

    # Syarat Akurasi Training & Testing
    y_pred_train = model.predict(X_train_vec)
    y_pred_test = model.predict(X_test_vec)

    acc_train = accuracy_score(y_train, y_pred_train) * 100
    acc_test = accuracy_score(y_test, y_pred_test) * 100

    print(f"Akurasi Training : {acc_train:.2f}%")
    print(f"Akurasi Testing  : {acc_test:.2f}%")
    print("-" * 60)

# Cetak Laporan Klasifikasi untuk Skema Terbaik (Deep Learning)
print("\nLaporan Klasifikasi Skema 3 (Deep Learning):")
print(classification_report(y_test, models["Skema 3 (Deep Learning - MLP)"].predict(X_test_vec)))

Melatih Skema 1 (Naive Bayes)...
Akurasi Training : 93.63%
Akurasi Testing  : 92.37%
------------------------------------------------------------
Melatih Skema 2 (Random Forest)...
Akurasi Training : 98.99%
Akurasi Testing  : 92.37%
------------------------------------------------------------
Melatih Skema 3 (Deep Learning - MLP)...
Akurasi Training : 98.71%
Akurasi Testing  : 91.74%
------------------------------------------------------------

Laporan Klasifikasi Skema 3 (Deep Learning):
              precision    recall  f1-score   support

     Negatif       0.79      0.78      0.79       321
      Netral       0.19      0.07      0.10        42
     Positif       0.95      0.97      0.96      1537

    accuracy                           0.92      1900
   macro avg       0.64      0.61      0.62      1900
weighted avg       0.91      0.92      0.91      1900



/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


In [36]:
# ==========================================
# SEL INFERENCE (PENGUJIAN MODEL TERBAIK)
# ==========================================

ulasan_baru = [
    "aplikasi ini sangat membantu, pesan tiket hotel gampang banget",
    "kecewa parah, minta refund susahnya minta ampun tidak ada kejelasan",
    "tampilannya lumayan, tapi kadang agak loading kalau sinyal jelek"
]

print("Hasil Inference dengan Skema 3 (Deep Learning):")
print("="*60)

# Gunakan model MLP yang sudah dilatih dari kotak sebelumnya
model_terbaik = models["Skema 3 (Deep Learning - MLP)"]

for teks in ulasan_baru:
    # Jangan lupa bersihkan dan transformasikan teksnya dulu
    teks_bersih = bersihkan_teks(teks)
    vektor_teks = vectorizer.transform([teks_bersih])
    prediksi = model_terbaik.predict(vektor_teks)

    print(f"Ulasan Asli : '{teks}'")
    print(f"Prediksi    : {prediksi[0]}")
    print("-" * 60)

Hasil Inference dengan Skema 3 (Deep Learning):
Ulasan Asli : 'aplikasi ini sangat membantu, pesan tiket hotel gampang banget'
Prediksi    : Positif
------------------------------------------------------------
Ulasan Asli : 'kecewa parah, minta refund susahnya minta ampun tidak ada kejelasan'
Prediksi    : Negatif
------------------------------------------------------------
Ulasan Asli : 'tampilannya lumayan, tapi kadang agak loading kalau sinyal jelek'
Prediksi    : Positif
------------------------------------------------------------


In [37]:
!pip freeze > requirements.txt